# LeetCode 826 — Most Profit Assigning Work bằng Python

Link đề: https://leetcode.com/problems/most-profit-assigning-work/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxProfitAssignment(self, difficulty: List[int], profit: List[int], worker: List[int]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên). Điểm mấu chốt: **lợi nhuận không tăng theo độ khó** (việc khó hơn có thể lương thấp hơn), nên mỗi công nhân phải lấy **lợi nhuận lớn nhất trong mọi việc dễ hơn hoặc bằng khả năng** của mình — không phải việc khó nhất làm được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` công việc `(difficulty[i], profit[i])`, `m` công nhân có khả năng `worker[j]`; mỗi công nhân làm **tối đa một** việc có độ khó `≤` khả năng, **một việc có thể được nhiều người làm**; tối đa hoá tổng lợi nhuận.
2. Nắm các công cụ Python: `sorted` với `zip`/`key`, `bisect_right`, tổng tiền tố kiểu "lớn nhất đến giờ" (**prefix maximum**, dùng `max` cộng dồn hoặc `itertools.accumulate(..., max)`), hai con trỏ, mảng đếm theo giá trị.
3. Đi từ **mỗi công nhân quét mọi việc (brute force)** → **sắp xếp việc + prefix max + nhị phân** → **sắp xếp cả công nhân + hai con trỏ** → **mảng theo độ khó `best[d]` + prefix max (không sắp xếp)**.
4. Hiểu mấu chốt: vì các công nhân **độc lập** (một việc làm nhiều lần được), tối ưu từng công nhân riêng lẻ cũng tối ưu toàn cục; mỗi công nhân chọn `max{profit[i] : difficulty[i] ≤ worker[j]}`.
5. Tránh các lỗi kinh điển: chọn việc **khó nhất** làm được (thay vì lương cao nhất), gán mỗi việc tối đa một lần, `bisect_left` thay `bisect_right` (bỏ sót việc bằng đúng khả năng), quên cập nhật prefix max khi hai con trỏ đi qua việc lương thấp, và quên rằng công nhân không làm được việc nào nhận `0`.

## 2. Đề bài & ví dụ

Có `n` công việc và `m` công nhân. Cho ba mảng:

- `difficulty[i]`, `profit[i]`: độ khó và lợi nhuận của việc thứ `i`;
- `worker[j]`: khả năng của công nhân `j` (chỉ làm được việc có độ khó **không vượt quá** `worker[j]`).

Mỗi công nhân được giao **tối đa một việc**, nhưng **một việc có thể được làm nhiều lần** (nhiều công nhân cùng làm một việc, mỗi người nhận đủ lợi nhuận). Công nhân không làm được việc nào thì lợi nhuận `0`.

Trả về **tổng lợi nhuận lớn nhất** có thể đạt.

**Ví dụ 1:** `difficulty = [2,4,6,8,10]`, `profit = [10,20,30,40,50]`, `worker = [4,5,6,7]` → `100`. Các công nhân làm việc độ khó `[4,4,6,6]` với lợi nhuận `[20,20,30,30]`.

**Ví dụ 2:** `difficulty = [85,47,57]`, `profit = [24,66,99]`, `worker = [40,25,25]` → `0` (không ai đủ khả năng).

**Ràng buộc:** `n == difficulty.length == profit.length`, `m == worker.length`; `1 <= n, m <= 10^4`; `1 <= difficulty[i], profit[i], worker[i] <= 10^5`.

In [1]:
import random
import time
from bisect import bisect_right, bisect_left
from itertools import accumulate, product
from typing import List

ex1 = ([2, 4, 6, 8, 10], [10, 20, 30, 40, 50], [4, 5, 6, 7])
ex2 = ([85, 47, 57], [24, 66, 99], [40, 25, 25])
for d, p, w in (ex1, ex2):
    print("việc (độ khó, lương):", list(zip(d, p)), "| công nhân:", w)

việc (độ khó, lương): [(2, 10), (4, 20), (6, 30), (8, 40), (10, 50)] | công nhân: [4, 5, 6, 7]
việc (độ khó, lương): [(85, 24), (47, 66), (57, 99)] | công nhân: [40, 25, 25]


## 3. Kiến thức nền

### 3.1 `zip`, sắp xếp theo khoá

`zip(difficulty, profit)` ghép từng cặp `(độ khó, lợi nhuận)`; `sorted(zip(...))` sắp theo độ khó (rồi theo lợi nhuận nếu hòa) — tuple so sánh từ trái sang phải. Ta dùng nó để đưa các việc về thứ tự tăng dần độ khó.

In [2]:
jobs = sorted(zip([85, 47, 57, 47], [24, 66, 99, 70]))
print(jobs)
print("độ khó:", [d for d, _ in jobs], "| lợi nhuận:", [p for _, p in jobs])

[(47, 66), (47, 70), (57, 99), (85, 24)]
độ khó: [47, 47, 57, 85] | lợi nhuận: [66, 70, 99, 24]


### 3.2 Prefix maximum ("lớn nhất đến giờ")

Cho dãy lợi nhuận đã xếp theo độ khó tăng dần `[10, 5, 30, 20]`, **prefix max** là `[10, 10, 30, 30]`: phần tử thứ `i` là lớn nhất của `profit[0..i]`. Nó trả lời ngay câu hỏi "trong **mọi** việc có độ khó ≤ x, lợi nhuận lớn nhất là bao nhiêu?" — dù lợi nhuận không đơn điệu. `itertools.accumulate(seq, max)` tính prefix max.

In [3]:
profits = [10, 5, 30, 20]
print(list(accumulate(profits, max)))
run, out = 0, []
for x in profits:
    run = max(run, x)
    out.append(run)
print(out, "(viết tay, cùng kết quả)")

[10, 10, 30, 30]
[10, 10, 30, 30] (viết tay, cùng kết quả)


### 3.3 `bisect_right`: đếm phần tử `≤ x` trong mảng đã sắp xếp

`bisect_right(a, x)` trả số phần tử `≤ x` của mảng tăng `a` (chỉ số chèn phía **phải** các phần tử bằng `x`). Với độ khó đã sắp, `bisect_right(difficulties, ability)` là **số việc làm được** (độ khó `≤` khả năng); việc cuối cùng làm được có chỉ số `bisect_right(...) − 1`. `bisect_left` thì **không** tính các việc có độ khó **bằng** đúng khả năng — lỗi hay gặp.

In [4]:
diffs = [2, 4, 4, 6, 8]
for ability in (1, 2, 4, 5, 8, 9):
    print(f"khả năng {ability}: bisect_right = {bisect_right(diffs, ability)} | bisect_left = {bisect_left(diffs, ability)}")

khả năng 1: bisect_right = 0 | bisect_left = 0
khả năng 2: bisect_right = 1 | bisect_left = 0
khả năng 4: bisect_right = 3 | bisect_left = 1
khả năng 5: bisect_right = 3 | bisect_left = 3
khả năng 8: bisect_right = 5 | bisect_left = 4
khả năng 9: bisect_right = 5 | bisect_left = 5


### 3.4 Hai con trỏ trên hai dãy đã sắp xếp

Nếu cả việc (theo độ khó) lẫn công nhân (theo khả năng) đều sắp tăng, thì khi duyệt công nhân từ yếu đến mạnh, tập các việc làm được chỉ **mở rộng** (không bao giờ thu hẹp). Một con trỏ `i` trên danh sách việc chỉ tiến, kèm biến `best` = lợi nhuận lớn nhất đã gặp, cho mỗi công nhân trong `O(1)` khấu hao.

### 3.5 Mảng đếm theo giá trị

Độ khó chỉ tới `10^5`. Ta có thể lập mảng `best[d]` (lợi nhuận lớn nhất của việc có độ khó **đúng** `d`) rồi lấy prefix max theo `d`: `best[d]` = lợi nhuận lớn nhất của mọi việc độ khó `≤ d`. Mỗi công nhân tra `best[ability]` trong `O(1)` mà không cần sắp xếp.

In [5]:
MAXV = 10
d, p = [2, 4, 6, 8, 10], [10, 20, 30, 40, 50]
best = [0] * (MAXV + 1)
for di, pi in zip(d, p):
    best[di] = max(best[di], pi)
print("theo độ khó đúng d :", best)
for x in range(1, MAXV + 1):
    best[x] = max(best[x], best[x - 1])
print("prefix max         :", best)
print("công nhân [4,5,6,7] nhận:", [best[a] for a in (4, 5, 6, 7)], "tổng", sum(best[a] for a in (4, 5, 6, 7)))

theo độ khó đúng d : [0, 0, 10, 0, 20, 0, 30, 0, 40, 0, 50]
prefix max         : [0, 0, 10, 10, 20, 20, 30, 30, 40, 40, 50]
công nhân [4,5,6,7] nhận: [20, 20, 30, 30] tổng 100


## 4. Ý tưởng nền tảng / trực giác

Mỗi công nhân **độc lập**: việc ông A làm không ngăn bà B làm cùng việc (việc làm được nhiều lần). Vậy mỗi người chỉ cần chọn việc **tốt nhất cho riêng mình**: trong các việc có độ khó `≤ khả năng`, lấy việc có **lợi nhuận lớn nhất**.

Ví dụ 1: việc `(độ khó, lương)`: `(2,10) (4,20) (6,30) (8,40) (10,50)`.

| công nhân | các việc làm được | lương lớn nhất |
|---|---|---|
| 4 | độ khó 2, 4 | 20 |
| 5 | độ khó 2, 4 | 20 |
| 6 | độ khó 2, 4, 6 | 30 |
| 7 | độ khó 2, 4, 6 | 30 |

Tổng `20 + 20 + 30 + 30 = 100` ✓.

**Bẫy:** việc khó hơn **không** nhất thiết lương cao hơn. Giả sử việc `(2, 100)` và `(9, 5)`; công nhân khả năng `9` nên chọn `(2, 100)`, không phải `(9, 5)` (việc khó nhất làm được).

In [6]:
jobs = [(2, 100), (9, 5)]
ability = 9
doable = [(d, p) for d, p in jobs if d <= ability]
print("việc làm được:", doable)
print("việc khó nhất làm được:", max(doable), "-> lương", max(doable)[1])
print("việc lương cao nhất   :", max(doable, key=lambda j: j[1]), "-> lương", max(p for _, p in doable))

việc làm được: [(2, 100), (9, 5)]
việc khó nhất làm được: (9, 5) -> lương 5
việc lương cao nhất   : (2, 100) -> lương 100


## 5. Cách 1 — Brute Force: mỗi công nhân quét mọi việc

**Ý tưởng:** với mỗi công nhân, duyệt tất cả việc, lấy lợi nhuận lớn nhất trong các việc có `difficulty[i] ≤ ability` (nếu không có việc nào thì `0`); cộng lại.

In [7]:
def profit_brute(difficulty, profit, worker):
    total = 0
    for ability in worker:
        best = 0
        for d, p in zip(difficulty, profit):
            if d <= ability and p > best:
                best = p
        total += best
    return total


print(profit_brute(*ex1), profit_brute(*ex2))
assert profit_brute(*ex1) == 100 and profit_brute(*ex2) == 0

100 0


**Độ phức tạp:** `O(n · m)` = `10^8` ở giới hạn lớn nhất (quá chậm trong Python); bộ nhớ `O(1)`.

**Nút thắt:** mỗi công nhân quét lại **toàn bộ** việc. Nhiều công nhân cùng "nhìn" cùng tập việc con; ta nên **chuẩn bị** một lần "với độ khó `≤ x`, lương lớn nhất là bao nhiêu" rồi tra cứu.

## 6. Ý tưởng thuật toán / bước đột phá

Định nghĩa hàm `g(x) = max{ profit[i] : difficulty[i] ≤ x }` (và `0` nếu rỗng). Đáp án là `Σ_j g(worker[j])`. Hàm `g` **không giảm** theo `x` (tập việc chỉ mở rộng) và là một **hàm bậc thang** thay đổi chỉ ở các độ khó của việc. Ta chỉ cần xây `g` một lần:

- **Cách A (sắp xếp + nhị phân):** sắp việc theo độ khó, tính prefix max của lợi nhuận; `g(x) = prefix_max[bisect_right(difficulties, x) − 1]`.
- **Cách B (hai con trỏ):** sắp cả công nhân; duyệt công nhân từ yếu đến mạnh, tiến con trỏ việc và cập nhật `best`.
- **Cách C (mảng theo giá trị):** lập `best[0..maxD]` bằng prefix max theo độ khó, tra `best[ability]`.

Cả ba đều tính cùng một hàm `g`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** sắp xếp + prefix max + nhị phân, **Cách 3** sắp xếp cả công nhân + hai con trỏ, **Cách 4** mảng theo độ khó + prefix max.

### Cách 2 — Sắp xếp việc, prefix max, nhị phân

**Giải thích từng đoạn:**

1. `jobs = sorted(zip(difficulty, profit))`; tách `diffs` (độ khó tăng) và `pm = list(accumulate(profits, max))`.
2. Với mỗi công nhân: `k = bisect_right(diffs, ability)` (số việc làm được). Nếu `k == 0` nhận `0`; ngược lại nhận `pm[k − 1]`.

In [8]:
def profit_bisect(difficulty, profit, worker):
    jobs = sorted(zip(difficulty, profit))
    diffs = [d for d, _ in jobs]
    pm = list(accumulate((p for _, p in jobs), max))
    total = 0
    for ability in worker:
        k = bisect_right(diffs, ability)
        if k:
            total += pm[k - 1]
    return total


print(profit_bisect(*ex1), profit_bisect(*ex2))
assert profit_bisect(*ex1) == 100 and profit_bisect(*ex2) == 0

100 0


In [9]:
# Code nộp LeetCode cho Cách 2
from bisect import bisect_right
from itertools import accumulate


class SolutionBisect:
    def maxProfitAssignment(self, difficulty: List[int], profit: List[int], worker: List[int]) -> int:
        jobs = sorted(zip(difficulty, profit))
        diffs = [d for d, _ in jobs]
        pm = list(accumulate((p for _, p in jobs), max))
        ans = 0
        for ability in worker:
            k = bisect_right(diffs, ability)
            if k:
                ans += pm[k - 1]
        return ans

**Độ phức tạp:** sắp xếp `O(n log n)`; mỗi công nhân `O(log n)`: tổng `O((n + m) log n)`; bộ nhớ `O(n)`.

### Cách 3 — Sắp xếp cả hai + hai con trỏ

**Giải thích từng đoạn:**

1. Sắp việc theo độ khó; sắp công nhân theo khả năng tăng dần.
2. `i = 0`, `best = 0`. Với mỗi công nhân (từ yếu tới mạnh): trong khi `i < n` và `jobs[i].difficulty ≤ ability`: `best = max(best, jobs[i].profit)`; `i += 1`. Cộng `best`.

`best` là lợi nhuận lớn nhất của **mọi việc đã đi qua** — chính là `g(ability)`; con trỏ không lùi vì công nhân sau mạnh hơn.

In [10]:
def profit_two_pointers(difficulty, profit, worker):
    jobs = sorted(zip(difficulty, profit))
    i, best, total = 0, 0, 0
    for ability in sorted(worker):
        while i < len(jobs) and jobs[i][0] <= ability:
            best = max(best, jobs[i][1])
            i += 1
        total += best
    return total


print(profit_two_pointers(*ex1), profit_two_pointers(*ex2))
assert profit_two_pointers(*ex1) == 100 and profit_two_pointers(*ex2) == 0

100 0


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionTwoPointers:
    def maxProfitAssignment(self, difficulty: List[int], profit: List[int], worker: List[int]) -> int:
        jobs = sorted(zip(difficulty, profit))
        i, best, ans = 0, 0, 0
        for ability in sorted(worker):
            while i < len(jobs) and jobs[i][0] <= ability:
                best = max(best, jobs[i][1])
                i += 1
            ans += best
        return ans

**Độ phức tạp:** hai lần sắp xếp `O(n log n + m log m)`, hai con trỏ `O(n + m)`; bộ nhớ `O(n)`. Con trỏ `i` chỉ tiến nên tổng thao tác tuyến tính.

### Cách 4 — Mảng theo độ khó + prefix max (không cần sắp xếp)

**Ý tưởng:** độ khó ≤ `10^5` nên dùng mảng độ dài `maxD + 1` thay cho sắp xếp: `best[d] = max(best[d], profit)` cho mỗi việc; rồi `best[x] = max(best[x], best[x−1])` (prefix max theo giá trị). Công nhân tra `best[min(ability, maxD)]`.

In [12]:
def profit_array(difficulty, profit, worker):
    top = max(difficulty)
    best = [0] * (top + 1)
    for d, p in zip(difficulty, profit):
        if p > best[d]:
            best[d] = p
    for x in range(1, top + 1):
        if best[x - 1] > best[x]:
            best[x] = best[x - 1]
    return sum(best[min(a, top)] for a in worker)


print(profit_array(*ex1), profit_array(*ex2))
assert profit_array(*ex1) == 100 and profit_array(*ex2) == 0

100 0


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionArray:
    def maxProfitAssignment(self, difficulty: List[int], profit: List[int], worker: List[int]) -> int:
        top = max(difficulty)
        best = [0] * (top + 1)
        for d, p in zip(difficulty, profit):
            best[d] = max(best[d], p)
        for x in range(1, top + 1):
            best[x] = max(best[x], best[x - 1])
        return sum(best[min(a, top)] for a in worker)

**Độ phức tạp:** `O(n + m + maxD)` thời gian (không có `log`), bộ nhớ `O(maxD)` (`10^5`). Nhanh nhất ở giới hạn đề; nhưng không phù hợp nếu giá trị độ khó rất lớn (`10^9`) — khi đó dùng Cách 2/3.

## 8. TRACE từng bước

**Trace Cách 3 (hai con trỏ)** với ví dụ 1: việc đã sắp `[(2,10),(4,20),(6,30),(8,40),(10,50)]`, công nhân đã sắp `[4,5,6,7]`:

| công nhân `a` | con trỏ `i` tiến qua | `best` sau | cộng vào tổng | tổng |
|---|---|---|---|---|
| 4 | `(2,10)`, `(4,20)` (dừng ở `(6,30)` vì `6 > 4`) | 20 | +20 | 20 |
| 5 | không tiến (`6 > 5`) | 20 | +20 | 40 |
| 6 | `(6,30)` (dừng ở `(8,40)`) | 30 | +30 | 70 |
| 7 | không tiến (`8 > 7`) | 30 | +30 | 100 |

Hàm trace in từng bước, kèm một ví dụ có lương **không đơn điệu** theo độ khó để thấy `best` không giảm:

In [14]:
def trace_two_pointers(difficulty, profit, worker):
    jobs = sorted(zip(difficulty, profit))
    print("việc đã sắp:", jobs)
    i, best, total = 0, 0, 0
    for ability in sorted(worker):
        moved = []
        while i < len(jobs) and jobs[i][0] <= ability:
            if jobs[i][1] > best:
                best = jobs[i][1]
            moved.append(jobs[i])
            i += 1
        total += best
        print(f"công nhân {ability:3d}: con trỏ đi qua {moved if moved else '-'} | best = {best} | tổng = {total}")
    return total


assert trace_two_pointers(*ex1) == 100
print()
assert trace_two_pointers([2, 5, 9, 12], [50, 20, 10, 60], [1, 6, 9, 20, 5]) == 0 + 50 + 50 + 50 + 60

việc đã sắp: [(2, 10), (4, 20), (6, 30), (8, 40), (10, 50)]
công nhân   4: con trỏ đi qua [(2, 10), (4, 20)] | best = 20 | tổng = 20
công nhân   5: con trỏ đi qua - | best = 20 | tổng = 40
công nhân   6: con trỏ đi qua [(6, 30)] | best = 30 | tổng = 70
công nhân   7: con trỏ đi qua - | best = 30 | tổng = 100

việc đã sắp: [(2, 50), (5, 20), (9, 10), (12, 60)]
công nhân   1: con trỏ đi qua - | best = 0 | tổng = 0
công nhân   5: con trỏ đi qua [(2, 50), (5, 20)] | best = 50 | tổng = 50
công nhân   6: con trỏ đi qua - | best = 50 | tổng = 100
công nhân   9: con trỏ đi qua [(9, 10)] | best = 50 | tổng = 150
công nhân  20: con trỏ đi qua [(12, 60)] | best = 60 | tổng = 210


**Trace Cách 4 (mảng)** cho cùng ví dụ nhỏ: `best` theo độ khó đúng `d` rồi prefix max; công nhân tra `best[ability]`. Hàm trace in hai mảng:

In [15]:
def trace_array(difficulty, profit, worker):
    top = max(difficulty)
    best = [0] * (top + 1)
    for d, p in zip(difficulty, profit):
        best[d] = max(best[d], p)
    print("theo độ khó đúng:", best)
    for x in range(1, top + 1):
        best[x] = max(best[x], best[x - 1])
    print("sau prefix max   :", best)
    got = [best[min(a, top)] for a in worker]
    print("lợi nhuận từng công nhân", list(zip(worker, got)), "-> tổng", sum(got))
    return sum(got)


assert trace_array(*ex1) == 100
assert trace_array([2, 5, 9], [50, 20, 10], [1, 6, 9, 20]) == 0 + 50 + 50 + 50

theo độ khó đúng: [0, 0, 10, 0, 20, 0, 30, 0, 40, 0, 50]
sau prefix max   : [0, 0, 10, 10, 20, 20, 30, 30, 40, 40, 50]
lợi nhuận từng công nhân [(4, 20), (5, 20), (6, 30), (7, 30)] -> tổng 100
theo độ khó đúng: [0, 0, 50, 0, 0, 20, 0, 0, 0, 10]
sau prefix max   : [0, 0, 50, 50, 50, 50, 50, 50, 50, 50]
lợi nhuận từng công nhân [(1, 0), (6, 50), (9, 50), (20, 50)] -> tổng 150


## 9. Code nộp LeetCode

Chọn **sắp xếp + hai con trỏ** (rõ ràng, không phụ thuộc giá trị lớn). Giữ nguyên `class Solution`, `maxProfitAssignment(self, difficulty, profit, worker)`:

In [16]:
from typing import List


class Solution:
    def maxProfitAssignment(self, difficulty: List[int], profit: List[int], worker: List[int]) -> int:
        jobs = sorted(zip(difficulty, profit))     # việc theo độ khó tăng dần
        i, best, ans = 0, 0, 0
        for ability in sorted(worker):             # công nhân yếu -> mạnh
            while i < len(jobs) and jobs[i][0] <= ability:
                best = max(best, jobs[i][1])       # lương lớn nhất trong mọi việc làm được
                i += 1
            ans += best
        return ans


print(Solution().maxProfitAssignment(*ex1), Solution().maxProfitAssignment(*ex2))
assert Solution().maxProfitAssignment(*ex1) == 100 and Solution().maxProfitAssignment(*ex2) == 0

100 0


## 10. Vì sao đúng

**(a) Các công nhân độc lập.** Ràng buộc duy nhất là "mỗi công nhân tối đa một việc, độ khó `≤` khả năng". Một việc có thể được làm nhiều lần, nên lựa chọn của công nhân này **không ảnh hưởng** tập lựa chọn của công nhân khác. Tổng lợi nhuận là tổng của `m` hàm riêng; cực đại hoá tổng = cực đại hoá từng số hạng: mỗi công nhân `j` nhận `g(worker[j]) = max{profit[i] : difficulty[i] ≤ worker[j]}` (hoặc `0`).

**(b) Hai con trỏ tính đúng `g`.** Bất biến: sau khi xử lý công nhân khả năng `a`, con trỏ `i` đã đi qua đúng các việc có độ khó `≤ a` (vì việc đã sắp tăng và vòng `while` dừng ngay khi gặp việc độ khó `> a`) và `best` là lương lớn nhất của chúng, tức `g(a)`. Công nhân kế (khả năng `a' ≥ a`) cần thêm đúng các việc độ khó trong `(a, a']`, nên tiếp tục từ `i` là đủ; `best` chỉ tăng nên vẫn là `g(a')`.

**(c) Cách 2 và 4.** `pm[k − 1]` với `k` = số việc độ khó `≤ a` chính là lương lớn nhất của chúng. Mảng `best` sau prefix max thoả `best[x] = max(best[x−1], lương của việc độ khó đúng x) = g(x)` quy nạp theo `x`.

**(d) Trường hợp không làm được.** Nếu không việc nào có độ khó `≤ a` thì `g(a) = 0` (công nhân nhận `0`), khớp `k = 0`/`best = 0` ban đầu.

Ta kiểm chứng bằng **vét cạn thật**: liệt kê mọi cách gán mỗi công nhân một việc (hoặc không việc) trên input nhỏ (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Công nhân × việc | `O(n·m)` | `O(1)` | oracle; `10^8` ở giới hạn lớn |
| 2. Sort + prefix max + bisect | `O((n + m) log n)` | `O(n)` | tổng quát |
| 3. Sort hai bên + hai con trỏ | `O(n log n + m log m)` | `O(n)` | **đáp án chính** |
| 4. Mảng theo độ khó | `O(n + m + maxD)` | `O(maxD)` | nhanh nhất khi `maxD ≤ 10^5` |

Đo thực tế ở giới hạn lớn nhất (`n = m = 10^4`, giá trị tới `10^5`):

In [17]:
def random_case(rng, n, m, hi=10 ** 5):
    return ([rng.randint(1, hi) for _ in range(n)], [rng.randint(1, hi) for _ in range(n)], [rng.randint(1, hi) for _ in range(m)])


def bench(fn, case, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*case)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(826)
big = random_case(rng, 10 ** 4, 10 ** 4)
ref = profit_two_pointers(*big)
for name, fn in [("bisect", profit_bisect), ("two_pointers", profit_two_pointers), ("array", profit_array)]:
    assert fn(*big) == ref
    print(f"{name:13s} {bench(fn, big) * 1000:8.2f} ms")
mid = random_case(rng, 2000, 2000)
t = bench(profit_brute, mid, 1)
print(f"brute (n = m = 2000): {t * 1000:8.1f} ms  -> n = m = 10^4 khoảng {t * 25:.1f} s")

bisect            6.55 ms
two_pointers      6.37 ms


array             7.94 ms


brute (n = m = 2000):    139.5 ms  -> n = m = 10^4 khoảng 3.5 s


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **vét cạn thật sự** (liệt kê mọi cách gán), (E) ngẫu nhiên có seed + tính chất, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "brute": profit_brute,
    "bisect": profit_bisect,
    "two_pointers": profit_two_pointers,
    "array": profit_array,
    "Solution": lambda d, p, w: Solution().maxProfitAssignment(list(d), list(p), list(w)),
    "SolutionBisect": lambda d, p, w: SolutionBisect().maxProfitAssignment(list(d), list(p), list(w)),
    "SolutionTwoPointers": lambda d, p, w: SolutionTwoPointers().maxProfitAssignment(list(d), list(p), list(w)),
    "SolutionArray": lambda d, p, w: SolutionArray().maxProfitAssignment(list(d), list(p), list(w)),
}


def check(d, p, w, want, names=None):
    for name in (names or IMPLS):
        got = IMPLS[name](d, p, w)
        assert got == want, (name, d, p, w, got, want)


check(*ex1, 100)
check(*ex2, 0)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 8 cài đặt đúng


### B. Edge case

- Một việc, một công nhân (đủ / không đủ khả năng).
- Khả năng **bằng đúng** độ khó (phải làm được, `<=`).
- Việc dễ lương cao, việc khó lương thấp (lương không đơn điệu).
- Nhiều việc **cùng độ khó** khác lương.
- Công nhân yếu hơn mọi việc → `0`; mọi công nhân giỏi hơn mọi việc → `m × (lương lớn nhất)`.
- Công nhân trùng khả năng.

In [19]:
cases = [
    ([5], [7], [5], 7), ([5], [7], [4], 0), ([5], [7], [6, 5, 4], 14),
    ([2, 9], [100, 5], [9], 100),                              # việc khó lương thấp
    ([4, 4, 4], [10, 30, 20], [4, 4], 60),                    # cùng độ khó: lấy lương lớn nhất, nhiều người làm
    ([10, 20], [1, 2], [1, 2, 3], 0),                          # ai cũng yếu
    ([1, 2, 3], [5, 9, 7], [100, 100, 100, 100], 36),         # ai cũng mạnh: 4 x 9
    ([3, 1, 2], [30, 10, 20], [1, 2, 3, 3], 10 + 20 + 30 + 30),   # thứ tự đầu vào không sắp
]
for d, p, w, want in cases:
    check(d, p, w, want)
print(len(cases), "edge case: tất cả cài đặt đúng")

8 edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_hardest(difficulty, profit, worker):            # chọn việc KHÓ NHẤT làm được, không phải lương cao nhất
    jobs = sorted(zip(difficulty, profit))
    total = 0
    for a in worker:
        doable = [j for j in jobs if j[0] <= a]
        if doable:
            total += doable[-1][1]
    return total


def bug_once(difficulty, profit, worker):               # mỗi việc chỉ giao một lần
    jobs = sorted(zip(difficulty, profit), key=lambda j: -j[1])
    used = [False] * len(jobs)
    total = 0
    for a in sorted(worker, reverse=True):
        for k, (d, p) in enumerate(jobs):
            if not used[k] and d <= a:
                used[k] = True
                total += p
                break
    return total


def bug_bisect_left(difficulty, profit, worker):        # bisect_left bỏ sót việc đúng bằng khả năng
    jobs = sorted(zip(difficulty, profit))
    diffs = [d for d, _ in jobs]
    pm = list(accumulate((p for _, p in jobs), max))
    total = 0
    for a in worker:
        k = bisect_left(diffs, a)
        if k:
            total += pm[k - 1]
    return total


def bug_no_best_update(difficulty, profit, worker):     # hai con trỏ nhưng best = lương việc cuối, không phải max
    jobs = sorted(zip(difficulty, profit))
    i, best, total = 0, 0, 0
    for a in sorted(worker):
        while i < len(jobs) and jobs[i][0] <= a:
            best = jobs[i][1]                           # SAI: không lấy max
            i += 1
        total += best
    return total


def bug_unsorted_workers(difficulty, profit, worker):  # hai con trỏ nhưng quên sắp xếp công nhân
    jobs = sorted(zip(difficulty, profit))
    i, best, total = 0, 0, 0
    for a in worker:                                    # SAI: thiếu sorted
        while i < len(jobs) and jobs[i][0] <= a:
            best = max(best, jobs[i][1])
            i += 1
        total += best
    return total


BUGS = [bug_hardest, bug_once, bug_bisect_left, bug_no_best_update, bug_unsorted_workers]
rng = random.Random(1)
tests = [random_case(rng, rng.randint(1, 6), rng.randint(1, 6), 10) for _ in range(600)]
for bug in BUGS:
    bad = next((t for t in tests if bug(*t) != profit_brute(*t)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên {bad}: ra {bug(*bad)}, đúng {profit_brute(*bad)}")

bug_hardest            sai trên ([2, 3, 5, 2, 6], [9, 7, 9, 4, 5], [5, 10, 8, 9, 7, 10]): ra 34, đúng 54
bug_once               sai trên ([2, 5], [2, 8], [8, 8, 7, 4, 2]): ra 10, đúng 28
bug_bisect_left        sai trên ([2, 5], [2, 8], [8, 8, 7, 4, 2]): ra 26, đúng 28
bug_no_best_update     sai trên ([2, 3, 5, 2, 6], [9, 7, 9, 4, 5], [5, 10, 8, 9, 7, 10]): ra 34, đúng 54
bug_unsorted_workers   sai trên ([2, 5], [2, 8], [8, 8, 7, 4, 2]): ra 40, đúng 28


### D. Vét cạn thật sự: liệt kê mọi cách gán

**Không dựa vào ý "tối ưu từng người":** với input nhỏ, mỗi công nhân chọn một trong `n + 1` lựa chọn (một việc, hoặc không làm), kiểm tra ràng buộc độ khó và lấy tổng lớn nhất trên **mọi** `(n+1)^m` cách gán. Đối chiếu với mọi cài đặt trên các input nhỏ (cả ngẫu nhiên và vét cạn mọi giá trị trong khoảng nhỏ).

In [21]:
def true_oracle(difficulty, profit, worker):
    n = len(difficulty)
    best = 0
    for assign in product(range(-1, n), repeat=len(worker)):          # -1: không làm
        total = 0
        ok = True
        for a, k in zip(worker, assign):
            if k >= 0:
                if difficulty[k] > a:
                    ok = False
                    break
                total += profit[k]
        if ok and total > best:
            best = total
    return best


assert true_oracle(*ex1[:2], [4, 5]) == 40 and true_oracle(*ex2) == 0
cnt = 0
# mọi input với n = 2 việc, m = 2 công nhân, giá trị 1..3
for d in product(range(1, 4), repeat=2):
    for p in product(range(1, 4), repeat=2):
        for w in product(range(1, 4), repeat=2):
            want = true_oracle(d, p, w)
            check(list(d), list(p), list(w), want)
            cnt += 1
print("n=2, m=2, giá trị 1..3:", cnt, "input vét cạn khớp")

rng = random.Random(20240826)
cnt2 = 0
for _ in range(1200):
    n, m = rng.randint(1, 4), rng.randint(1, 4)
    case = random_case(rng, n, m, 6)
    check(*case, true_oracle(*case))
    cnt2 += 1
print("n, m <= 4, giá trị <= 6:", cnt2, "input ngẫu nhiên khớp oracle liệt kê mọi cách gán")

n=2, m=2, giá trị 1..3: 729 input vét cạn khớp


n, m <= 4, giá trị <= 6: 1200 input ngẫu nhiên khớp oracle liệt kê mọi cách gán


### E. Ngẫu nhiên có seed và tính chất

In [22]:
rng = random.Random(20240827)
count = 0
for _ in range(1500):
    n, m = rng.randint(1, 60), rng.randint(1, 60)
    hi = rng.choice([5, 20, 100, 10 ** 5])
    case = random_case(rng, n, m, hi)
    want = profit_brute(*case)
    for name in IMPLS:
        assert IMPLS[name](*case) == want, (name, case)
    count += 1
print(count, "input ngẫu nhiên: mọi cài đặt trùng brute")

# tính chất: (1) hoán vị thứ tự việc / công nhân không đổi; (2) thêm công nhân không giảm; (3) tăng lương không giảm; (4) tăng khả năng không giảm
for _ in range(300):
    d, p, w = random_case(rng, rng.randint(1, 10), rng.randint(1, 10), 30)
    base = Solution().maxProfitAssignment(list(d), list(p), list(w))
    idx = list(range(len(d))); rng.shuffle(idx)
    ws = list(w); rng.shuffle(ws)
    assert Solution().maxProfitAssignment([d[i] for i in idx], [p[i] for i in idx], ws) == base
    assert Solution().maxProfitAssignment(list(d), list(p), list(w) + [rng.randint(1, 30)]) >= base
    p2 = list(p); k = rng.randrange(len(p2)); p2[k] += 5
    assert Solution().maxProfitAssignment(list(d), p2, list(w)) >= base
    w2 = list(w); j = rng.randrange(len(w2)); w2[j] += 7
    assert Solution().maxProfitAssignment(list(d), list(p), w2) >= base
print("tính chất đơn điệu và bất biến hoán vị: OK trên 300 input")

1500 input ngẫu nhiên: mọi cài đặt trùng brute
tính chất đơn điệu và bất biến hoán vị: OK trên 300 input


### F. Giới hạn lớn nhất

`n = m = 10^4`, giá trị tới `10^5`: ngẫu nhiên; mọi việc cùng độ khó; mọi công nhân cùng khả năng; việc có lương giảm dần theo độ khó (lương không đơn điệu); tổng có thể tới `10^4 × 10^5 = 10^9`.

In [23]:
rng = random.Random(777)
n = m = 10 ** 4
fast = ["bisect", "two_pointers", "array", "Solution", "SolutionBisect", "SolutionTwoPointers", "SolutionArray"]
cases = {
    "ngẫu nhiên": random_case(rng, n, m),
    "cùng độ khó": ([50000] * n, [rng.randint(1, 10 ** 5) for _ in range(n)], [rng.randint(1, 10 ** 5) for _ in range(m)]),
    "cùng khả năng": ([rng.randint(1, 10 ** 5) for _ in range(n)], [rng.randint(1, 10 ** 5) for _ in range(n)], [70000] * m),
    "lương giảm theo độ khó": (list(range(1, n + 1)), list(range(10 ** 5, 10 ** 5 - n, -1)), [rng.randint(1, 10 ** 5) for _ in range(m)]),
    "tất cả cực đại": ([10 ** 5] * n, [10 ** 5] * n, [10 ** 5] * m),
}
for name, case in cases.items():
    want = profit_array(*case)
    # đối chiếu độc lập trên 300 công nhân đầu: g(a) tính bằng cách duyệt việc theo lương GIẢM dần (không prefix max)
    jobs = sorted(zip(case[0], case[1]), key=lambda j: -j[1])
    sample = case[2][:300]
    ref = sum(next((p_ for d_, p_ in jobs if d_ <= a), 0) for a in sample)
    assert profit_two_pointers(case[0], case[1], sample) == ref
    for impl in fast:
        assert IMPLS[impl](*case) == want, (name, impl)
    print(f"{name:24s}: đáp án {want:>12d} (7 cài đặt khớp)")
assert profit_array(*cases["tất cả cực đại"]) == 10 ** 4 * 10 ** 5

ngẫu nhiên              : đáp án    998224152 (7 cài đặt khớp)


cùng độ khó             : đáp án    495080196 (7 cài đặt khớp)
cùng khả năng           : đáp án    999950000 (7 cài đặt khớp)
lương giảm theo độ khó  : đáp án   1000000000 (7 cài đặt khớp)


tất cả cực đại          : đáp án   1000000000 (7 cài đặt khớp)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chọn việc **khó nhất** làm được

Nghĩ rằng "việc khó hơn thì lương cao hơn". Ví dụ 2 và trường hợp `(2, 100)`, `(9, 5)` cho thấy điều đó sai.

In [24]:
d, p, w = [2, 9], [100, 5], [9]
print("khó nhất:", bug_hardest(d, p, w), "| đúng:", Solution().maxProfitAssignment(d, p, w))
assert bug_hardest(d, p, w) == 5 and Solution().maxProfitAssignment(d, p, w) == 100

khó nhất: 5 | đúng: 100


**Vì sao sai:** lương không đơn điệu theo độ khó. **Đúng:** lấy `max` lợi nhuận trong **mọi** việc làm được (prefix max).

### Lỗi 2: giao mỗi việc tối đa một lần

Đề nói rõ "một việc có thể được làm nhiều lần". Nếu giao mỗi việc một người (như bài phân công thường gặp), tổng nhỏ hơn.

In [25]:
print("mỗi việc một lần:", bug_once(*ex1), "| đúng:", Solution().maxProfitAssignment(*ex1))
assert bug_once(*ex1) < 100

mỗi việc một lần: 60 | đúng: 100


**Vì sao sai:** ở ví dụ 1 hai công nhân cùng làm việc `(4, 20)` và hai người cùng làm `(6, 30)`; bản sai chỉ cho mỗi việc một người, nên có công nhân nhận việc lương thấp hơn. **Đúng:** các công nhân độc lập.

### Lỗi 3: `bisect_left` thay cho `bisect_right`

Công nhân có khả năng **bằng đúng** độ khó của việc vẫn làm được (`≤`). `bisect_left` đếm các việc độ khó `<` khả năng nên bỏ sót việc bằng.

In [26]:
d, p, w = [5], [7], [5]
print("bisect_left:", bug_bisect_left(d, p, w), "| đúng:", Solution().maxProfitAssignment(d, p, w))
print("ví dụ 1:", bug_bisect_left(*ex1), "| đúng:", Solution().maxProfitAssignment(*ex1))
assert bug_bisect_left(d, p, w) == 0 and bug_bisect_left(*ex1) != 100

bisect_left: 0 | đúng: 7
ví dụ 1: 80 | đúng: 100


**Vì sao sai:** khả năng 5, độ khó 5: `bisect_left(diffs, 5) = 0` nên không có việc nào. **Đúng:** `bisect_right`.

### Lỗi 4: quên lấy `max` khi hai con trỏ đi qua việc

Gán `best = jobs[i][1]` thay vì `max(best, jobs[i][1])` làm `best` giảm khi gặp việc khó hơn nhưng lương thấp hơn.

In [27]:
d, p, w = [1, 2], [10, 3], [5]
print("không max:", bug_no_best_update(d, p, w), "| đúng:", Solution().maxProfitAssignment(d, p, w))
assert bug_no_best_update(d, p, w) == 3 and Solution().maxProfitAssignment(d, p, w) == 10

không max: 3 | đúng: 10


**Vì sao sai:** sau khi đi qua `(2, 3)`, `best` bị ghi đè thành `3`. **Đúng:** `best = max(best, jobs[i][1])`.

### Lỗi 5: hai con trỏ nhưng không sắp xếp công nhân

Con trỏ `i` chỉ tiến; nếu công nhân mạnh đứng **trước** công nhân yếu, công nhân yếu nhận `best` của công nhân mạnh (cao hơn khả năng thật).

In [28]:
d, p, w = [1, 10], [5, 100], [10, 1]
print("không sắp công nhân:", bug_unsorted_workers(d, p, w), "| đúng:", Solution().maxProfitAssignment(d, p, w))
assert bug_unsorted_workers(d, p, w) == 200 and Solution().maxProfitAssignment(d, p, w) == 105

không sắp công nhân: 200 | đúng: 105


**Vì sao sai:** công nhân khả năng `10` đẩy con trỏ qua cả việc `(10, 100)`; công nhân khả năng `1` sau đó nhận `100` dù chỉ làm được việc độ khó `1`. **Đúng:** `for ability in sorted(worker)`.

### Lỗi 6 (bonus): quên công nhân không làm được việc nào

Phải trả `0` cho họ (không phải lương nhỏ nhất, không phải lỗi). Khởi tạo `best = 0` và đảm bảo không cộng bậy; `bisect_right(...) == 0` thì không cộng.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "nhiều đối tượng độc lập, mỗi đối tượng chọn phần tử tốt nhất trong một tập con được xác định bởi ngưỡng" → xây **hàm bậc thang** `g(x) = max{giá trị : khoá ≤ x}` một lần (sắp xếp + prefix max) rồi trả lời từng truy vấn bằng **nhị phân** hoặc **hai con trỏ**.

**Mẹo:**

1. Hỏi: các công nhân có tranh nhau việc không? Nếu **không** (việc làm được nhiều lần) thì tối ưu riêng lẻ.
2. Prefix max biến dữ liệu **không đơn điệu** thành đơn điệu để nhị phân/hai con trỏ dùng được.
3. Khi giá trị nhỏ (`≤ 10^5`), mảng theo giá trị bỏ cả `log`.
4. `bisect_right` cho điều kiện `≤`, `bisect_left` cho `<`.
5. Hai con trỏ cần **cả hai dãy cùng chiều** (đã sắp xếp).
6. Dùng **vét cạn gán nhãn** `(n+1)^m` làm oracle trên input nhỏ.

**Họ bài cùng pattern:** Assign Cookies (455), Maximum Profit in Job Scheduling (1235, khó hơn), Successful Pairs of Spells and Potions (2300), Maximum Matching of Players With Trainers (2410), Boats to Save People (881).

## 15. Biến thể

1. **Mỗi việc chỉ làm một lần** (bài phân công / khớp cặp có trọng số): khác hẳn, cần chọn việc theo lương giảm dần và kiểm tra khả năng ghép (Hall) — hoặc đặt thành bài khớp.
2. **Trả về việc mà mỗi công nhân làm** (bài tập 5).
3. **Có chi phí thuê** công nhân: trừ chi phí và bỏ những ai lỗ.
4. **Nhiều tiêu chí** (khoảng độ khó `[lo, hi]`): cần cấu trúc nâng cao (cây phân đoạn).
5. **Truy vấn trực tuyến** (công nhân đến từng người): xây một lần `g`, trả lời `O(log n)`.

Cài đặt biến thể 5 (xây `g` một lần, dùng cho nhiều truy vấn) và đối chiếu:

In [29]:
class ProfitOracle:
    """Xây g(x) = lương lớn nhất của việc có độ khó <= x một lần; mỗi truy vấn O(log n)."""

    def __init__(self, difficulty, profit):
        jobs = sorted(zip(difficulty, profit))
        self.diffs = [d for d, _ in jobs]
        self.pm = list(accumulate((p for _, p in jobs), max))

    def query(self, ability):
        k = bisect_right(self.diffs, ability)
        return self.pm[k - 1] if k else 0


rng = random.Random(15)
d, p, w = random_case(rng, 50, 200, 100)
orc = ProfitOracle(d, p)
assert sum(orc.query(a) for a in w) == profit_brute(d, p, w)
for a in (0, 1, 50, 100, 10 ** 9):
    brute_g = max([pp for dd, pp in zip(d, p) if dd <= a], default=0)
    assert orc.query(a) == brute_g
print("ProfitOracle: query(ability) khớp brute; ví dụ g(50) =", orc.query(50), ", g(100) =", orc.query(100))

ProfitOracle: query(ability) khớp brute; ví dụ g(50) = 93 , g(100) = 97


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Công nhân × việc | Mỗi công nhân quét mọi việc | `O(n·m)` | `O(1)` | Oracle; input nhỏ |
| 2. Sort + prefix max + bisect | `g(x)` bằng nhị phân | `O((n+m) log n)` | `O(n)` | Tổng quát; truy vấn trực tuyến |
| 3. Sort hai bên + hai con trỏ | Con trỏ việc chỉ tiến, `best` cộng dồn | `O(n log n + m log m)` | `O(n)` | **Đáp án chính**, an toàn |
| 4. Mảng theo độ khó | `best[d]` + prefix max theo giá trị | `O(n + m + maxD)` | `O(maxD)` | Giá trị nhỏ (`≤ 10^5`); nhanh nhất |

Cả bốn khớp nhau và khớp oracle liệt kê mọi cách gán (mục 12D).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [30]:
d, p, w = [1, 3, 5], [4, 2, 9], [2, 3, 4, 6]
print(sorted(zip(d, p)))
print(list(accumulate([4, 2, 9], max)))
print(bisect_right([1, 3, 5], 3), bisect_left([1, 3, 5], 3), bisect_right([1, 3, 5], 0))
print(Solution().maxProfitAssignment(d, p, w))

[(1, 4), (3, 2), (5, 9)]
[4, 4, 9]
2 1 0
21


### Bài 2 (tính tay)

`difficulty = [3, 1, 5, 2]`, `profit = [6, 2, 4, 8]`, `worker = [1, 2, 3, 4, 5]`.

- (a) Sắp xếp việc theo độ khó, tính prefix max của lợi nhuận.
- (b) Mỗi công nhân nhận bao nhiêu? Tổng bao nhiêu?
- (c) Nếu bỏ prefix max và chỉ lấy lương của việc **khó nhất** làm được, tổng là bao nhiêu và sai ở đâu?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai con trỏ:

In [31]:
def two_pointers_todo(difficulty, profit, worker):
    jobs = sorted(zip(difficulty, profit))
    i, best, total = 0, 0, 0
    for ability in sorted(worker):
        # TODO 1: trong khi còn việc và độ khó <= ability: cập nhật best = max(best, lương), tiến i
        # TODO 2: cộng best vào total
        pass
    return total


# Khi làm xong, bỏ comment để kiểm tra:
# assert two_pointers_todo(*ex1) == 100 and two_pointers_todo(*ex2) == 0

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng sắp xếp + prefix max + `bisect_right`:

In [32]:
class MySolution:
    def maxProfitAssignment(self, difficulty, profit, worker):
        pass            # TODO


# Khi làm xong:
# assert MySolution().maxProfitAssignment(*ex1) == 100
# assert MySolution().maxProfitAssignment(*ex2) == 0

### Bài 5 (mini challenge): chỉ rõ việc của từng công nhân

Viết `assign(difficulty, profit, worker)` trả về `(tổng, danh sách)` với danh sách có `m` phần tử: chỉ số **việc** (theo vị trí gốc trong `difficulty`) mà mỗi công nhân làm, hoặc `-1` nếu không làm được việc nào. Tổng phải bằng đáp án tối ưu và việc phải hợp lệ (độ khó `≤` khả năng). Nếu nhiều việc cùng lương lớn nhất, chọn việc bất kỳ (kiểm bằng hàm hợp lệ, không so cứng).

In [33]:
def assign(difficulty, profit, worker):
    pass            # TODO


# Khi làm xong (kiểm bằng hàm hợp lệ):
# total, jobs = assign(*ex1)
# assert total == 100 and all(j == -1 or ex1[0][j] <= a for j, a in zip(jobs, ex1[2]))

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `sorted(zip(d, p))` sắp theo độ khó.
- Mức 2: prefix max của `[4, 2, 9]` là `[4, 4, 9]`.
- Mức 3: `bisect_right([1,3,5], 3) = 2`, `bisect_left = 1`, `bisect_right(…, 0) = 0`. Công nhân `2,3,4,6` nhận `4, 4, 4, 9` → `21`.

**Bài 2**

- Mức 1: việc sắp: `(1,2), (2,8), (3,6), (5,4)`.
- Mức 2: prefix max lương: `2, 8, 8, 8`.
- Mức 3: công nhân `1,2,3,4,5` nhận `2, 8, 8, 8, 8` → `34`. Nếu chỉ lấy việc khó nhất: `2, 8, 6, 6, 4` → `26` (sai vì bỏ qua việc lương `8`).

**Bài 3**

- Mức 1: giống mục 7 Cách 3.
- Mức 2: `while i < len(jobs) and jobs[i][0] <= ability`.
- Mức 3: trong vòng: `best = max(best, jobs[i][1]); i += 1`; sau vòng: `total += best`.

**Bài 4**

- Mức 1: `jobs = sorted(zip(...))`, `pm = list(accumulate(..., max))`.
- Mức 2: `k = bisect_right(diffs, ability)`.
- Mức 3: `ans += pm[k-1] if k else 0`.

**Bài 5**

- Mức 1: lưu kèm chỉ số gốc khi sắp xếp.
- Mức 2: prefix max kèm chỉ số việc đạt max.
- Mức 3: công nhân tra `k = bisect_right`; nếu `k == 0` trả `-1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [34]:
d, p, w = [1, 3, 5], [4, 2, 9], [2, 3, 4, 6]
assert sorted(zip(d, p)) == [(1, 4), (3, 2), (5, 9)]
assert list(accumulate([4, 2, 9], max)) == [4, 4, 9]
assert (bisect_right([1, 3, 5], 3), bisect_left([1, 3, 5], 3), bisect_right([1, 3, 5], 0)) == (2, 1, 0)
assert Solution().maxProfitAssignment(d, p, w) == 4 + 4 + 4 + 9 == 21
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) Việc sắp: `(1,2), (2,8), (3,6), (5,4)`; prefix max `2, 8, 8, 8`. (b) Công nhân `1..5` nhận `2, 8, 8, 8, 8`: tổng `34`. (c) Việc khó nhất làm được: `2, 8, 6, 6, 4` → `26`: sai vì công nhân khả năng `3` bỏ qua việc `(2, 8)` lương cao hơn `(3, 6)`.

In [35]:
d, p, w = [3, 1, 5, 2], [6, 2, 4, 8], [1, 2, 3, 4, 5]
jobs = sorted(zip(d, p))
assert jobs == [(1, 2), (2, 8), (3, 6), (5, 4)]
assert list(accumulate([q for _, q in jobs], max)) == [2, 8, 8, 8]
assert Solution().maxProfitAssignment(d, p, w) == 34 == profit_brute(d, p, w)
assert bug_hardest(d, p, w) == 26
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [36]:
def two_pointers_answer(difficulty, profit, worker):
    jobs = sorted(zip(difficulty, profit))
    i, best, total = 0, 0, 0
    for ability in sorted(worker):
        while i < len(jobs) and jobs[i][0] <= ability:
            best = max(best, jobs[i][1])
            i += 1
        total += best
    return total


assert two_pointers_answer(*ex1) == 100 and two_pointers_answer(*ex2) == 0
rng = random.Random(3)
for _ in range(300):
    case = random_case(rng, rng.randint(1, 8), rng.randint(1, 8), 12)
    assert two_pointers_answer(*case) == profit_brute(*case)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def maxProfitAssignment(self, difficulty, profit, worker):
        jobs = sorted(zip(difficulty, profit))
        diffs = [d for d, _ in jobs]
        pm = list(accumulate((p for _, p in jobs), max))
        total = 0
        for a in worker:
            k = bisect_right(diffs, a)
            total += pm[k - 1] if k else 0
        return total


assert MySolutionAnswer().maxProfitAssignment(*ex1) == 100 and MySolutionAnswer().maxProfitAssignment(*ex2) == 0
rng = random.Random(4)
for _ in range(300):
    case = random_case(rng, rng.randint(1, 8), rng.randint(1, 8), 12)
    assert MySolutionAnswer().maxProfitAssignment(*case) == profit_brute(*case)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Sắp xếp kèm chỉ số gốc; prefix max lưu **chỉ số gốc của việc đạt max**.

In [38]:
def assign_answer(difficulty, profit, worker):
    order = sorted(range(len(difficulty)), key=lambda i: difficulty[i])
    diffs = [difficulty[i] for i in order]
    best_idx = []
    cur = -1
    for i in order:
        if cur == -1 or profit[i] > profit[cur]:
            cur = i
        best_idx.append(cur)
    jobs, total = [], 0
    for a in worker:
        k = bisect_right(diffs, a)
        if k == 0:
            jobs.append(-1)
        else:
            j = best_idx[k - 1]
            jobs.append(j)
            total += profit[j]
    return total, jobs


def valid_assignment(difficulty, profit, worker, total, jobs):
    return (len(jobs) == len(worker)
            and all(j == -1 or difficulty[j] <= a for j, a in zip(jobs, worker))
            and total == sum(profit[j] for j in jobs if j != -1))


total, jobs = assign_answer(*ex1)
print(total, jobs)
assert total == 100 and valid_assignment(*ex1, total, jobs)
rng = random.Random(5)
for _ in range(500):
    d, p, w = random_case(rng, rng.randint(1, 8), rng.randint(1, 8), 12)
    t, js = assign_answer(d, p, w)
    assert valid_assignment(d, p, w, t, js) and t == profit_brute(d, p, w)
total, jobs = assign_answer(*ex2)
assert total == 0 and jobs == [-1, -1, -1]
print("Bài 5 OK: phân công hợp lệ và tổng tối ưu trên 500 input")

100 [1, 1, 2, 2]
Bài 5 OK: phân công hợp lệ và tổng tối ưu trên 500 input


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (mỗi đối tượng chọn max trong tập con theo ngưỡng — hai con trỏ):**

```python
class Solution:
    def maxProfitAssignment(self, difficulty, profit, worker):
        jobs = sorted(zip(difficulty, profit))
        i, best, ans = 0, 0, 0
        for ability in sorted(worker):
            while i < len(jobs) and jobs[i][0] <= ability:
                best = max(best, jobs[i][1])
                i += 1
            ans += best
        return ans
```

**Template (nhị phân):** `k = bisect_right(diffs, ability); ans += pm[k-1] if k else 0` với `pm = accumulate(lương, max)`.

**Checklist trước khi submit:**

- [ ] Mỗi công nhân lấy **lương lớn nhất** trong mọi việc làm được (không phải việc khó nhất)?
- [ ] Một việc được làm **nhiều lần**; không dùng "mỗi việc một người"?
- [ ] Điều kiện `độ khó ≤ khả năng` (bằng nhau vẫn làm được): `bisect_right` / `<=`?
- [ ] Hai con trỏ: sắp xếp **cả hai** dãy; `best = max(best, ...)` không bị ghi đè?
- [ ] Công nhân không làm được việc nào nhận `0`?
- [ ] Đã thử ví dụ 1 (100), ví dụ 2 (0), `[(2,100),(9,5)]`, khả năng đúng bằng độ khó?
- [ ] Đúng tên `class Solution`, `maxProfitAssignment(self, difficulty, profit, worker)`, không in debug.